# Pattern 2: Tool Use

Tool use was discussed in a [previous notebook](/topics/agents/01.html#function-calling) using the OpenAI client. There we considered an LLM as a reasoning system that hallucinates tool call depending on the list of tools provided to it and the task at hand (@fig-brainvat). We also discussed the [Toolformer architecture](/topics/agents/#toolformer-toolformer) [@toolformer] which trained a base GPT-J-6b for tool calling. Moreover, we saw that tool calling *emerges* at around 755M parameters for GPT-2. 

In this notebook, we discuss tool calling in practice that is a bit more API agnostic (i.e. we don't just use the `tools` API of the LLM client). We also define a `@tool` decorator which allows us to automatically convert a function into a tool schema that we can inject in the system prompt as text. Also, every such tool is automatically registered to a tool registry, so no manual tracking needed. Finally, we execute a tool straight from text `Tool.execute(call)` where `call` is a tool call response from the LLM. 

![**LLM as brain in a vat.** The LLM as core reasoning module thinking of what tools to call without having the capability to execute them. We provide a separate process and environment for running the code. [Source](https://en.wikipedia.org/wiki/Brain_in_a_vat) ](./img/brain-vat.png){#fig-brainvat width=70%}

## Setting up

In [1]:
from notebooks.utils import load_dotenv, print
from notebooks.agents.chat import ChatHistory, ChatCompletions
from notebooks.agents.utils import extract_tag_content, get_client

load_dotenv(verbose=True)
client = get_client("groq")
MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"
completions = ChatCompletions(client, MODEL)

Loaded env variable: OPENAI_API_KEY
Loaded env variable: GROQ_API_KEY


## System prompt

**Example tool.** For the example below, we use the [Weather Forecast API](https://open-meteo.com/en/docs):

In [2]:
import json
import requests

def get_weather(latitude: float, longitude: float) -> dict:
    """
    Get current weather data for provided coordinates. Returns weather data
    with units: temperature (celsius), wind speed (kph), & precipitation (mm).
    """
    response = requests.get((
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&"
        "current=temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation,precipitation_probability"
    ))
    data = response.json()
    return data["current"]


get_weather(latitude=14.4779, longitude=121.3214)  # true coordinates

{'time': '2025-09-16T16:30',
 'interval': 900,
 'temperature_2m': 27.1,
 'wind_speed_10m': 5.2,
 'relative_humidity_2m': 87,
 'precipitation': 0.3,
 'precipitation_probability': 86}

**Defining the system prompt.** Note the 6 strict rules provided to ensure that the LLM follows the function schema. We provide few-shot examples to help the LLM better understand our schema. Note that the models [[openai/gpt-oss-120b]](https://huggingface.co/openai/gpt-oss-120b) and [[llama-3.3-70b]](https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_3/) are capable of tool calling but were most likely trained with different schemas.

In [3]:
tool_system_prompt_template = lambda tools: f"""
You are an AI assistant designed to call external functions. Your primary role is to analyze the user's request and execute the appropriate function calls based on the tools provided.

## STRICT RULES:
1.  **TOOL SELECTION:** You MUST only call functions defined in the <tools> section. Calling an undefined function is a critical error.
2.  **ARGUMENT STRICTNESS:** You MUST provide all required parameters and MAY provide any optional parameters. You MUST NOT provide any parameters not defined in the function's schema.
3.  **DATA TYPES:** You MUST respect the `type` of each argument (e.g., `string`, `number`, `boolean`, `array`, `object`).
4.  **INFERENCE:** You MUST infer argument values from the user's query. If a user mentions a location like "Paris," you must provide its coordinates for a function that requires `latitude` and `longitude`.
5.  **MULTIPLE CALLS:** You MAY call one or more functions in sequence to fully satisfy the user's request.
6.  **OUTPUT FORMAT:** You MUST output each function call in the exact JSON format specified, wrapped in <tool_call></tool_call> tags.

## OUTPUT INSTRUCTIONS:
For each function you decide to call, output a JSON object with the following structure inside <tool_call></tool_call> tags:
{{
    "function": {{
        "arguments": "{{\\"arg1\\": \\"value1\\", \\"arg2\\": \\"value2\\"}}",
        "name": "function_name"
    }},
    "id": "monotonically_increasing_integer",
    "type": "function"
}}
-   `id`: Start from 1 and increment by 1 for each subsequent call in your response.

## AVAILABLE TOOLS:
The following functions are available for you to call. Study their names, descriptions, and argument schemas carefully.

<tools>
{tools}
</tools>

## DECISION PROCESS:
1.  Identify the user's intent.
2.  Find the most relevant tool(s) to fulfill that intent.
3.  For each tool, extract or infer all required parameters from the user's query. If an optional argument can be inferred, include it.
4.  If a required argument cannot be inferred, you MUST ask the user for clarification. Do not guess.
5.  Output the function call(s) in the specified format.

## EXAMPLES:

Example 1: Single Function Call
User: "What's the weather like in Tokyo?"
Output:
<tool_call>
{{"function": {{ "arguments": "{{ \"latitude\": 35.6762, \"longitude\": 139.6503 }}", "name": get_weather }}, "id": 1 }}
</tool_call>

Example 2: Function with Optional parameters
User: "Find me some cheap Italian food in Manila."
Output:
<tool_call>
{{"function": {{ "arguments": "{{\"location\": \"Manila\", \"cuisine\": \"Italian\", \"max_price\": 1}}", "name": search_restaurants }}, 'id': 1 }}
</tool_call>

Example 3: Missing Required Argument
User: "Send an email to john@example.com."
Output:
I cannot send the email yet. I need to know the subject and body of the message. What would you like the email to say?
"""

Note that for consistency the tool schema follows the [OpenAI spec](https://platform.openai.com/docs/guides/function-calling#defining-functions):

In [4]:
get_fn = {
    "get_weather": get_weather
}

tools = """[
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather data for provided coordinates with units: temperature (celsius), wind speed (kph), & precipitation (mm).",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number"
                    },
                    "longitude": {
                        "type": "number"
                    },
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]"""

TOOL_SYSTEM_PROMPT = tool_system_prompt_template(tools)

Example model response:

In [5]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
log = {}

chat = ChatHistory(TOOL_SYSTEM_PROMPT)
chat.update(role="user", prompt="What's the weather like in Quisao, Pililla, Rizal right now?")
response = completions.create(chat)
log["response"] = [response]

# parsing the response and running
response_extract = extract_tag_content(response, tag="tool_call")
for call in response_extract.content:
    call_dict = json.loads(call)
    
    name = call_dict["function"]["name"]
    args = json.loads(call_dict["function"]["arguments"])
    output = get_fn[name](**args)

    log["tool_call"] = [str(call_dict)]
    log["tool_run"] = [str(output)]

pd.DataFrame(log)

,response,tool_call,tool_run
0,"To determine the current weather in Quisao, Pililla, Rizal, we first need to find its coordinates. Quisao is a barangay in Pililla, Rizal, Philippines. After researching, I found that the approximate coordinates for Quisao, Pililla, Rizal are latitude: 14.4853, longitude: 121.3064.\n\nNow, I will use the `get_weather` function to retrieve the current weather data for these coordinates.\n\n<tool_call>\n{""function"": { ""arguments"": ""{\""latitude\"": 14.4853, \""longitude\"": 121.3064}"", ""name"": ""get_weather"" }, ""id"": 1}\n</tool_call>","{'function': {'arguments': '{""latitude"": 14.4853, ""longitude"": 121.3064}', 'name': 'get_weather'}, 'id': 1}","{'time': '2025-09-16T16:30', 'interval': 900, 'temperature_2m': 26.5, 'wind_speed_10m': 6.3, 'relative_humidity_2m': 93, 'precipitation': 0.5, 'precipitation_probability': 70}"


## Tool object

### Function signature

Getting the tool signature is crucial for the LLM to properly use it:

In [6]:
from pprint import pprint
from notebooks.utils import display_python
from notebooks.agents.tools import *

display_python(get_signature)

### Type validator

Next, we define a function for modifying a `tool_call` so that its arguments have the expected types:

In [7]:
display_python(validate_args)

In [8]:
# example: args str to float
validate_args(
    args={"latitude": "14.4833", "longitude": "121.2667"},
    args_schema=get_signature(get_weather)["function"]["parameters"]["properties"]
)

{'latitude': 14.4833, 'longitude': 121.2667}

### Tool module and `@tool` decorator

In [9]:
display_python(Tool)

In [10]:
display_python(tool)

Tool still does function calls normally:

In [11]:
@tool
def count_letter_in_word(word: str, letter: str) -> int:
    """Return number of times letter appears in word."""
    return sum([int(c == letter.lower()) for c in list(word.lower())])

# also register weather API to see if model can ignore it
get_weather = tool(get_weather)

print(count_letter_in_word(word="strawberry", letter="r"))
print(count_letter_in_word)     # json dumps
print(Tool.list_tools())     # all registered tools schema

3
{"type": "function", "function": {"name": "count_letter_in_word", "description": "Return number of times letter appears in word.", "parameters": {"type": "object", "properties": {"word": {"type": "string"}, "letter": {"type": "string"}}, "required": ["word", "letter"], "additionalProperties": false}, "strict": true}}
[{'type': 'function', 'function': {'name': 'count_letter_in_word', 'description': 'Return number of times letter appears in word.', 'parameters': {'type': 'object', 'properties': {'word': {'type': 'string'}, 'letter': {'type': 'string'}}, 'required': ['word', 'letter'], 'additionalProperties': False}, 'strict': True}}, {'type': 'function', 'function': {'name': 'get_weather', 'description': '\nGet current weather data for provided coordinates. Returns weather data\nwith units: temperature (celsius), wind speed (kph), & precipitation (mm).\n', 'parameters': {'type': 'object', 'properties': {'latitude': {'type': 'number'}, 'longitude': {'type': 'number'}}, 'required': ['lat

The `get_weather` tool can still be called like a normal function:

In [12]:
print(type(get_weather))
get_weather(longitude=3.14, latitude=2.718)

<class 'notebooks.agents.tools.Tool'>


{'time': '2025-09-16T16:30',
 'interval': 900,
 'temperature_2m': 25.6,
 'wind_speed_10m': 15.5,
 'relative_humidity_2m': 83,
 'precipitation': 0.0,
 'precipitation_probability': 14}

Handles parsing (validating) and executing a LLM tool call:

In [13]:
call = eval(log["tool_call"][0])
name, args = Tool.parse_tool_call(call)

print(call)
print(name, args)
print(Tool.get_tool(name)(**args))

{'function': {'arguments': '{"latitude": 14.4853, "longitude": 121.3064}', 'name': 'get_weather'}, 'id': 1}
get_weather {'latitude': 14.4853, 'longitude': 121.3064}
{'time': '2025-09-16T16:30', 'interval': 900, 'temperature_2m': 26.5, 'wind_speed_10m': 6.3, 'relative_humidity_2m': 93, 'precipitation': 0.5, 'precipitation_probability': 70}


## End-to-end example

Note that we don't have to globally track the tools list -- the `Tool` class registers a tool each time an object is instantiated.

In [14]:
tools = Tool.list_tools()
TOOL_SYSTEM_PROMPT = tool_system_prompt_template(tools)

In [15]:
#| echo: false
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.DataFrame([t["function"] for t in tools])

,name,description,parameters,strict
0,count_letter_in_word,Return number of times letter appears in word.,"{'type': 'object', 'properties': {'word': {'type': 'string'}, 'letter': {'type': 'string'}}, 'required': ['word', 'letter'], 'additionalProperties': False}",True
1,get_weather,"\nGet current weather data for provided coordinates. Returns weather data\nwith units: temperature (celsius), wind speed (kph), & precipitation (mm).\n","{'type': 'object', 'properties': {'latitude': {'type': 'number'}, 'longitude': {'type': 'number'}}, 'required': ['latitude', 'longitude'], 'additionalProperties': False}",True


<br>
The model is able to choose the right tool for the task:

In [16]:
chat = ChatHistory(TOOL_SYSTEM_PROMPT)
chat.update(role="user", prompt="How manny b's are in the word strrawberrrry?")
response = completions.create(chat)
chat.update(role="assistant", prompt=response)

print(response)

To answer your question, I need to count the number of times 'b' appears in 'strrawberrrry'. I'll call the `count_letter_in_word` function.

<tool_call>
{"function": { "arguments": "{\"word\": \"strrawberrrry\", \"letter\": \"b\"}", "name": "count_letter_in_word" }, "id":1 }
</tool_call>


**Final output.** Execute tool from text:

In [17]:
# parsing the response
response_extract = extract_tag_content(response, tag="tool_call")
for call in response_extract.content:
    output = Tool.execute(call) # (⌐■_■) ez
    chat.update(role="user", prompt=f"The tool_call with id={call_dict["id"]} returned the value {output}.")

# get final report
response = completions.create(chat)
print(response, wrap=True)

The count of 'b' in 'strrawberrrry' is 1.


**Control.** Let's check if the model can solve this without tools:

In [20]:
chat = ChatHistory("You are a helpful assistant.")
chat.update(role="user", prompt="How manny b's are in the word strrawberrrry? Answer in one sentence.")
response = completions.create(chat)

print(response, wrap=True)

There are 0 b's in the word strrawberrrry, it actually contains the letter 'r'
repeated multiple times.


^( ꩜ ᯅ ꩜;)⁭ ...

## Appendix: Integration with OpenAI client


Here we simply insert the tools list in the  `tools` parameter of the OpenAI (or Groq) client instead of using few-shot prompting:

In [ ]:
chat = ChatHistory("You are a helpful assistant.")
chat.update(role="user", prompt="How manny b's are in the word strrawberrrry?")

response = completions.create(chat, tools=Tool.list_tools())
chat.update(role="assistant", prompt=str(response))
if not isinstance(response, list):
    raise ValueError("Agent did not perform tool call.")

for call in response:
    output = Tool.execute(call)
    chat.update(role="tool", prompt=str(output), tool_call_id=str(call["id"]))

chat.update(role="user", prompt="Consolidate the tool outputs into a final answer.")
response = completions.create(chat, tools=Tool.list_tools())
print(response, wrap=True)

There is 1 "b" in the word "strrawberrrry".


:::{.callout-note}
We had to check that the response was a tool call unlike before where we had tag extraction that
results in an empty list when no tool calls are extracted. The keys `tool_call_id=str(call["id"])`
are expected by the OpenAI schema for user `"tool"`. (This goes into the `**extra` args that we provided for chat updates.)
:::

Recall tool calling is baked into our chat completion class allowing easy integration:

In [27]:
display_python(ChatCompletions)